In [ ]:
# Install required Google Cloud packages (commented out as these are typically one-time setup commands)
!pip install gcloud
!gcloud auth application-default login

# Import necessary Python libraries
import pandas as pd                # Data manipulation and analysis
import numpy as np                 # Numerical computing
import time                        # Time-related functions
import os                          # Operating system interfaces
import pandas_gbq                  # Pandas integration with BigQuery
from google.cloud import bigquery  # BigQuery client library
import glob                        # File path pattern matching
import openpyxl                    # Excel file handling
import csv                         # CSV file handling
import re                          # Regular expressions

# Note: The actual imports remain exactly as in the original code

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.4/454.4 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gcloud: filename=gcloud-0.18.3-py3-none-any.whl size=602927 sha256=8e4303523340fb7ec80b9cc7785a47cc9260732527af9d306d566ea43468de31
  Stored in directory: /root/.cache/pip/wheels/2a/62/75/3d74209bfebb8805823ae74afa28653aa1ea76d8b5a9d741ff
Successfully built gcloud
Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=FNjbemxJQ9bJCzvDIfLparikRm9dOZ&prompt=consent&token_usage=remote&access_type=offline&code_chal

# O Ano de 2024

In [ ]:
df = pd.read_excel('/content/Base_Estadic_2024(2).xlsx', sheet_name='Agropecuária', nrows=29, usecols=['Cod UF', 'Eagr01','Eagr03','Eagr04', 'Eagr05', 'Eagr06'])
df

,Cod UF,Eagr01,Eagr03,Eagr04,Eagr05,Eagr06
0,11,Não informou,Não informou,Não informou,Não informou,Não informou
1,12,Secretaria estadual exclusiva,Masculino,51,Branca,Ensino superior completo
2,13,Secretaria estadual exclusiva,Masculino,39,Preta,Mestrado
3,14,Secretaria estadual em conjunto com outras pol...,Masculino,51,Branca,Mestrado
4,15,Secretaria estadual em conjunto com outras pol...,Masculino,78,Branca,Ensino superior completo
5,16,Secretaria estadual exclusiva,Masculino,39,Parda,Especialização
6,17,Secretaria estadual exclusiva,Masculino,58,Branca,Ensino médio (2º Grau) completo
7,21,Secretaria estadual exclusiva,Masculino,41,Parda,Especialização
8,22,Secretaria estadual exclusiva,Masculino,51,Preta,Ensino superior completo
9,23,Setor subordinado a outra secretaria,Masculino,61,Parda,Ensino médio (2º Grau) completo


In [ ]:
uf = pd.read_excel('/content/Base_Estadic_2020.xlsx', sheet_name = 'Variáveis externas', usecols=[1,2,3])
uf

,Código da Unidade da Federação,Sigla da Unidade da Federação,Nome da Unidade da Federação
0,11,RO,Rondônia
1,12,AC,Acre
2,13,AM,Amazonas
3,14,RR,Roraima
4,15,PA,Pará
5,16,AP,Amapá
6,17,TO,Tocantins
7,21,MA,Maranhão
8,22,PI,Piauí
9,23,CE,Ceará


Renomeando as colunas

In [ ]:
df.columns

Index(['Cod UF', 'Eagr01', 'Eagr03', 'Eagr04', 'Eagr05', 'Eagr06'], dtype='object')

In [ ]:
df= df.rename(columns={'Cod UF':'cod_uf',
                       'Eagr01':'caracterizacao_orgao_gestor',
                        'Eagr03':'genero',
                        'Eagr04':'idade',
                        'Eagr05':'cor_raca',
                        'Eagr06':'grau_instrucao'})

In [ ]:
df['grau_instrucao'].unique()

array(['Não informou', 'Ensino superior completo', 'Mestrado',
       'Especialização', 'Ensino médio (2º Grau) completo',
       'Ensino médio (2º Grau) incompleto', 'Doutorado'], dtype=object)

In [ ]:
df['ano']=2024 #adicionando a coluna ano

In [ ]:
x= uf.pivot_table(columns=('Código da Unidade da Federação', 'Sigla da Unidade da Federação', 'Nome da Unidade da Federação'), aggfunc='size')


In [ ]:
uf = pd.DataFrame(x).reset_index()[['Código da Unidade da Federação', 'Sigla da Unidade da Federação', 'Nome da Unidade da Federação']]

In [ ]:
df = df.merge(uf, right_on='Código da Unidade da Federação',left_on='cod_uf') #juntando os dataframes, adicionando sigla e nome das UFs


In [ ]:
df = df.drop(['Código da Unidade da Federação'], axis=1) #eliminando coluna repetida

In [ ]:
df = df.rename(columns={'Sigla da Unidade da Federação':'sigla_uf',
                        'Nome da Unidade da Federação':'uf'}) #padronizando as colunas

In [ ]:
limites = [0, 30, 50, 65, 100]
categorias = ['Entre 18-29', 'Entre 30-49', 'Entre 50-64', 'Acima de 65']

idade_num = pd.to_numeric(df['idade'], errors='coerce')  # "Não informou" -> NaN

df['faixa_etaria'] = pd.cut(idade_num, bins=limites, labels=categorias, right=False)
df['faixa_etaria'] = df['faixa_etaria'].cat.add_categories('Não informou').fillna('Não informou')

In [ ]:
# criando dicionário
dict_esco = {'Ensino fundamental (1º Grau) incompleto':'Até Ensino Fundamental',
             'Ensino médio (2º Grau) completo':'Até Ensino Médio',
             'Ensino médio (2º Grau) incompleto':'Até Ensino Médio',
             'Ensino superior incompleto':'Até Ensino Superior Completo',
             'Ensino superior completo':'Até Ensino Superior Completo',
             'Especialização':'Até Pós Graduação ou Mestrado',
             'Mestrado':'Até Pós Graduação ou Mestrado',
             'Doutorado':'Até Doutorado'}

In [ ]:
df = df.replace({'grau_instrucao':dict_esco})

In [ ]:
df['grau_instrucao'].unique()

array(['Não informou', 'Até Ensino Superior Completo',
       'Até Pós Graduação ou Mestrado', 'Até Ensino Médio',
       'Até Doutorado'], dtype=object)

In [ ]:
df['caracterizacao_orgao_gestor']=df['caracterizacao_orgao_gestor'].str.title()

In [ ]:
df= df[['ano', 'sigla_uf','cod_uf', 'uf', 'caracterizacao_orgao_gestor','genero', 'faixa_etaria', 'cor_raca','grau_instrucao']]

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   ano                          27 non-null     int64   
 1   sigla_uf                     27 non-null     object  
 2   cod_uf                       27 non-null     int64   
 3   uf                           27 non-null     object  
 4   caracterizacao_orgao_gestor  27 non-null     object  
 5   genero                       27 non-null     object  
 6   faixa_etaria                 27 non-null     category
 7   cor_raca                     27 non-null     object  
 8   grau_instrucao               27 non-null     object  
dtypes: category(1), int64(2), object(6)
memory usage: 2.0+ KB


In [ ]:
df.head(27)

,ano,sigla_uf,cod_uf,uf,caracterizacao_orgao_gestor,genero,faixa_etaria,cor_raca,grau_instrucao
0,2024,RO,11,Rondônia,Não Informou,Não informou,Não informou,Não informou,Não informou
1,2024,AC,12,Acre,Secretaria Estadual Exclusiva,Masculino,Entre 50-64,Branca,Até Ensino Superior Completo
2,2024,AM,13,Amazonas,Secretaria Estadual Exclusiva,Masculino,Entre 30-49,Preta,Até Pós Graduação ou Mestrado
3,2024,RR,14,Roraima,Secretaria Estadual Em Conjunto Com Outras Pol...,Masculino,Entre 50-64,Branca,Até Pós Graduação ou Mestrado
4,2024,PA,15,Pará,Secretaria Estadual Em Conjunto Com Outras Pol...,Masculino,Acima de 65,Branca,Até Ensino Superior Completo
5,2024,AP,16,Amapá,Secretaria Estadual Exclusiva,Masculino,Entre 30-49,Parda,Até Pós Graduação ou Mestrado
6,2024,TO,17,Tocantins,Secretaria Estadual Exclusiva,Masculino,Entre 50-64,Branca,Até Ensino Médio
7,2024,MA,21,Maranhão,Secretaria Estadual Exclusiva,Masculino,Entre 30-49,Parda,Até Pós Graduação ou Mestrado
8,2024,PI,22,Piauí,Secretaria Estadual Exclusiva,Masculino,Entre 50-64,Preta,Até Ensino Superior Completo
9,2024,CE,23,Ceará,Setor Subordinado A Outra Secretaria,Masculino,Entre 50-64,Parda,Até Ensino Médio


# Consumindo a base anterior para agregar o novo ano

In [ ]:


query = """SELECT * FROM `repositoriodedadosgpsp.cargos_lideranca.ESTADIC_perfil_gestor_agropecuaria_tipo_orgao`"""
# Execute the query using pandas_gbq.read_gbq and load the result into a pandas DataFrame called 'df'.
# The 'project_id' specifies the Google Cloud Project to use.
df_old = pandas_gbq.read_gbq(query, project_id='repositoriodedadosgpsp')



Downloading: 100%|██████████|


In [ ]:
df_final = pd.concat([df, df_old], ignore_index=True)

In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   ano                          54 non-null     Int64 
 1   sigla_uf                     54 non-null     object
 2   cod_uf                       54 non-null     Int64 
 3   uf                           54 non-null     object
 4   caracterizacao_orgao_gestor  54 non-null     object
 5   genero                       54 non-null     object
 6   faixa_etaria                 54 non-null     object
 7   cor_raca                     54 non-null     object
 8   grau_instrucao               54 non-null     object
dtypes: Int64(2), object(7)
memory usage: 4.0+ KB


# Subindo para o GBQ

In [ ]:
# Define the BigQuery table schema with Portuguese descriptions
schema=[bigquery.SchemaField('ano','INTEGER',description='Ano da apuração daquele dado'),
        bigquery.SchemaField('sigla_uf','STRING',description='Sigla da UF'),
        bigquery.SchemaField('cod_uf','INTEGER',description='Código do IBGE da UF'),
        bigquery.SchemaField('uf','STRING',description='Nome da UF'),
        bigquery.SchemaField('caracterizacao_orgao_gestor','STRING',description='Caracterização do órgão no qual o gestor está'),
        bigquery.SchemaField('genero','STRING',description='Gênero autodeclarado ou não'),
        bigquery.SchemaField('faixa_etaria','STRING',description='faixa etária da observação'),
        bigquery.SchemaField('cor_raca','STRING',description='Raça/cor da pessoa observada'),
        bigquery.SchemaField('grau_instrucao','STRING',description='Escolaridade da pessoa ou do vínculo observado com detalhamento na pós-graduação')
        ]

# Initialize BigQuery client connection
client = bigquery.Client(project='repositoriodedadosgpsp')

# Create reference to target dataset
dataset_ref = client.dataset('cargos_lideranca')

# Create reference to target table with standardized naming convention:
# FONTE_algo_intuitivo_dado (MUNIC_quantidade_vinculos_mapa_v1)
table_ref = dataset_ref.table('ESTADIC_perfil_gestor_agropecuaria_tipo_orgao_v2')

# Configure the load job with our schema definition
job_config = bigquery.LoadJobConfig(
    schema=schema,
    # Optional parameters (commented out):
    # write_disposition="WRITE_TRUNCATE",  # Overwrites table if exists
    # create_disposition="CREATE_IF_NEEDED"  # Default behavior
)

# Execute the load job to upload DataFrame to BigQuery
job = client.load_table_from_dataframe(
    dataframe=df_final,
    destination=table_ref,
    job_config=job_config
)

# Wait for the job to complete
job.result()

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


LoadJob<project=repositoriodedadosgpsp, location=US, id=1eda09d3-4f32-4bec-ba85-20f285751c09>